In [0]:
from pyspark.sql import functions as F

country_currency_map = spark.createDataFrame([
    ("United States", "USD"), ("Canada", "CAD"), ("United Kingdom", "GBP"),
    ("France", "EUR"), ("Germany", "EUR"), ("Japan", "JPY"), ("Argentina", "BRL"),
    ("Brazil", "BRL"), ("Australia", "AUD"), ("China", "CNY"), ("India", "INR"),
    ("Mexico", "MXN"), ("South Korea", "KRW"), ("South Africa", "ZAR"),
    ("Switzerland", "CHF"), ("Sweden", "SEK"), ("Norway", "NOK"), ("Denmark", "DKK"),
    ("Poland", "PLN"), ("Turkey", "TRY"), ("Indonesia", "IDR"), ("Malaysia", "MYR"),
    ("Philippines", "PHP"), ("Thailand", "THB"), ("Israel", "ILS"), ("Hungary", "HUF"),
    ("Czechia", "CZK"), ("Romania", "RON"), ("Iceland", "ISK"), ("Singapore", "SGD"),
    ("Hong Kong", "HKD"), ("New Zealand", "NZD"),
], ["country_name", "transaction_currency"])

display(country_currency_map)

In [0]:
geo_df = spark.table("fraud_detection.silver.transactions_geo_enriched")

unmapped = (
    geo_df.select("country_name").distinct()
    .join(country_currency_map, on="country_name", how="left_anti")
)
display(unmapped)
print(f"Distinct countries in data: {geo_df.select('country_name').distinct().count()}")
print(f"Unmapped countries: {unmapped.count()}")

In [0]:
import requests, re

url = "https://raw.githubusercontent.com/thiagodp/country-to-currency/master/index.ts"
resp = requests.get(url)
resp.raise_for_status()

# parse lines like:  US: 'USD',
pattern = re.compile(r"^\s*([A-Z]{2}):\s*'([A-Z]{3})',?\s*$", re.MULTILINE)
matches = pattern.findall(resp.text)

print(f"Parsed {len(matches)} country-currency pairs")
print(matches[:5])

In [0]:
from pyspark.sql import functions as F

country_currency_df = spark.createDataFrame(matches, ["country_iso_code", "transaction_currency"])

country_currency_df.write.mode("overwrite").saveAsTable("fraud_detection.bronze.country_currency_map_raw")

display(country_currency_df.limit(10))
print(f"Row count: {country_currency_df.count()}")

In [0]:
geo_df = spark.table("fraud_detection.silver.transactions_geo_enriched")

geo_with_currency = geo_df.join(
    country_currency_df,
    on="country_iso_code",
    how="left"
)

unmapped_count = geo_with_currency.filter(F.col("transaction_currency").isNull()).count()
print(f"Transactions with no currency mapped: {unmapped_count}")